# rozan-f1

Colab notebook: MediaPipe Face Landmarker on Rozan test images.
Runs in Google Colab with Drive mounted. Dataset stays on Drive, not in this repo.


In [ ]:
!pip install ultralytics -q
!pip install mediapipe opencv-python-headless matplotlib -q


In [ ]:
from google.colab import drive
from ultralytics import YOLO  # kept from the original Colab session
drive.mount('/content/drive')


In [ ]:
import os
import urllib.request
import cv2
import matplotlib.pyplot as plt
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

model_url = 'https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task'
model_path = 'face_landmarker.task'
if not os.path.exists(model_path):
    print('Downloading face_landmarker.task...')
    urllib.request.urlretrieve(model_url, model_path)
    print('Download complete!')

base_options = python.BaseOptions(model_asset_path=model_path)
options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    output_face_blendshapes=False,
    output_facial_transformation_matrixes=False,
    num_faces=1,
)
detector = vision.FaceLandmarker.create_from_options(options)


In [ ]:
input_dir = '/content/drive/MyDrive/dataset/test_images'
output_dir = os.path.join(input_dir, 'processed_outputs')
os.makedirs(output_dir, exist_ok=True)
valid_extensions = ('.jpg', '.jpeg', '.png', '.webp')

image_files = [f for f in os.listdir(input_dir) if f.lower().endswith(valid_extensions)]
print(f'Found {len(image_files)} images to process...')

for filename in image_files:
    img_path = os.path.join(input_dir, filename)
    try:
        image_mp = mp.Image.create_from_file(img_path)
        detection_result = detector.detect(image_mp)
        img_cv = cv2.imread(img_path)
        h, w, _ = img_cv.shape
        if detection_result.face_landmarks:
            for face_landmarks in detection_result.face_landmarks:
                for lm in face_landmarks:
                    cx, cy = int(lm.x * w), int(lm.y * h)
                    cv2.circle(img_cv, (cx, cy), 2, (0, 0, 255), -1)
        output_path = os.path.join(output_dir, f'annotated_{filename}')
        cv2.imwrite(output_path, img_cv)
        print(f'Successfully processed and saved: {filename}')
    except Exception as e:
        print(f'Error processing {filename}: {e}')

print(f'All images processed! Results saved in: {output_dir}')
